# 01 — Data Scraper: Redfin + Census ACS

**Purpose**: Scrape sold-home listings from Redfin's GIS-CSV endpoint for the Philadelphia
suburban counties (Delaware, Montgomery, Chester, Bucks) and enrich with
Census ACS 5-year socioeconomic data at the ZCTA (zip code) level.

**Key design decisions**:
- Price-band pagination keeps each API call under Redfin's 350-result cap.
- **Automatic band splitting**: any band that hits the cap is re-scraped with
  narrower sub-bands — no manual "patch" cells needed.
- Exponential back-off with jitter absorbs transient rate limits / 429s.
- Census API key is read from the environment variable `CENSUS_API_KEY`
  (never hard-coded in source control).

**Outputs** → `data/redfin_sold_homes.csv`, `data/census_zcta.csv`,
`data/redfin_with_census.csv`

In [1]:
# ---------------------------------------------------------------------------
# Imports & Configuration
# ---------------------------------------------------------------------------
import requests
import pandas as pd
import numpy as np
import io
import os
import time
import random
import logging
from typing import List, Tuple

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
log = logging.getLogger(__name__)

SEED = 51
random.seed(SEED)
np.random.seed(SEED)

# ── Redfin scraper settings ─────────────────────────────────────────────────
COUNTIES = {
    "Delaware County":   {"region_id": 2383, "region_type": 5},
    "Montgomery County": {"region_id": 2406, "region_type": 5},
    "Chester County":    {"region_id": 2375, "region_type": 5},
    "Bucks County":      {"region_id": 2369, "region_type": 5},
}

SOLD_WITHIN_DAYS   = 365
MAX_PRICE          = 1_200_000
FINE_BAND_CEILING  = 700_000
FINE_BAND_WIDTH    = 10_000
COARSE_BAND_WIDTH  = 25_000
NUM_HOMES_PER_CALL = 350
REQUEST_DELAY      = 2.0        # base delay (seconds)
MAX_RETRIES        = 4          # per-request retry limit
MIN_SPLIT_WIDTH    = 2_500      # narrowest sub-band before giving up
PROPERTY_TYPES     = "1,2,3,4"

OUTPUT_DIR  = "data"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "redfin_sold_homes.csv")

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.redfin.com/",
}

BASE_URL = "https://www.redfin.com/stingray/api/gis-csv"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}/")

Output directory: data/


In [2]:
# ---------------------------------------------------------------------------
# Build price bands
# ---------------------------------------------------------------------------
def build_price_bands() -> List[Tuple[int, int]]:
    """
    Construct non-overlapping price bands that keep each Redfin call under
    the 350-result cap.

    Strategy
    --------
    - $0–200k   : single catch-all band (low density above the floor)
    - $200k–700k: $10k bands (highest density zone)
    - $700k–1.2M: $25k bands (density drops)
    """
    bands = [(0, 200_000)]

    price = 200_000
    while price < FINE_BAND_CEILING:
        bands.append((price, price + FINE_BAND_WIDTH))
        price += FINE_BAND_WIDTH

    price = FINE_BAND_CEILING
    while price < MAX_PRICE:
        bands.append((price, price + COARSE_BAND_WIDTH))
        price += COARSE_BAND_WIDTH

    total_calls = len(bands) * len(COUNTIES)
    est_min = total_calls * REQUEST_DELAY / 60
    log.info(
        f"Band plan: {len(bands)} bands × {len(COUNTIES)} counties "
        f"= {total_calls} calls (~{est_min:.0f} min at {REQUEST_DELAY}s delay)"
    )
    return bands

In [3]:
# ---------------------------------------------------------------------------
# Fetch a single band with exponential back-off
# ---------------------------------------------------------------------------
def fetch_band(
    region_id: int,
    region_type: int,
    min_price: int,
    max_price: int,
) -> pd.DataFrame:
    """
    Single Redfin API call.  On HTTP 429 / 5xx / timeout, retries up to
    MAX_RETRIES times with exponential back-off + jitter.
    Returns a DataFrame (possibly empty).
    """
    params = {
        "al": 1, "market": "philadelphia",
        "num_homes": NUM_HOMES_PER_CALL, "ord": "redfin-recommended-asc",
        "page_number": 1, "region_id": region_id, "region_type": region_type,
        "sf": "1,2,3,5,6,7", "sold_within_days": SOLD_WITHIN_DAYS,
        "sp": "true", "status": 9, "uipt": PROPERTY_TYPES, "v": 8,
        "min_price": min_price, "max_price": max_price,
    }

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            r = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=30)

            if r.status_code == 429 or r.status_code >= 500:
                wait = REQUEST_DELAY * (2 ** attempt) + random.uniform(0, 2)
                log.warning(
                    f"  HTTP {r.status_code} — retry {attempt}/{MAX_RETRIES} "
                    f"in {wait:.1f}s  (region {region_id}, "
                    f"${min_price:,}–${max_price:,})"
                )
                time.sleep(wait)
                continue

            r.raise_for_status()

            text = r.content.decode("utf-8")
            if text.startswith("{}&&"):
                text = text[4:].lstrip("\n")

            df = pd.read_csv(io.StringIO(text))
            return df

        except requests.exceptions.RequestException as e:
            wait = REQUEST_DELAY * (2 ** attempt) + random.uniform(0, 2)
            log.warning(f"  Request error ({e}) — retry {attempt}/{MAX_RETRIES} in {wait:.1f}s")
            time.sleep(wait)

    log.error(f"  FAILED after {MAX_RETRIES} retries: region {region_id}, ${min_price:,}–${max_price:,}")
    return pd.DataFrame()

In [4]:
# ---------------------------------------------------------------------------
# Scrape all counties with automatic band splitting on cap hits
# ---------------------------------------------------------------------------
def scrape_band_recursive(
    region_id: int,
    region_type: int,
    min_price: int,
    max_price: int,
    county_name: str,
) -> List[pd.DataFrame]:
    """
    Fetch a single band; if it hits the 350-cap AND the band is wide
    enough to split, bisect and recurse.  This eliminates all manual
    'patch' steps.
    """
    df = fetch_band(region_id, region_type, min_price, max_price)
    time.sleep(REQUEST_DELAY + random.uniform(0, 1))

    if df.empty:
        return []

    if len(df) >= NUM_HOMES_PER_CALL:
        width = max_price - min_price
        if width <= MIN_SPLIT_WIDTH:
            log.warning(
                f"  ⚠  Cap hit at minimum band width ${min_price:,}–${max_price:,} "
                f"({len(df)} rows) — accepting partial results"
            )
            df["county"] = county_name
            return [df]

        mid = min_price + width // 2
        log.info(f"  ↳ Splitting ${min_price:,}–${max_price:,} → two sub-bands at ${mid:,}")
        left  = scrape_band_recursive(region_id, region_type, min_price, mid, county_name)
        right = scrape_band_recursive(region_id, region_type, mid, max_price, county_name)
        return left + right

    df["county"] = county_name
    return [df]


def scrape_all_counties(price_bands: List[Tuple[int, int]]) -> pd.DataFrame:
    all_frames = []

    for county_name, cfg in COUNTIES.items():
        rid, rtype = cfg["region_id"], cfg["region_type"]
        log.info(f"\n{'=' * 55}")
        log.info(f"  {county_name}  (region_id={rid})")
        log.info(f"{'=' * 55}")

        county_frames = []
        for i, (lo, hi) in enumerate(price_bands):
            frames = scrape_band_recursive(rid, rtype, lo, hi, county_name)
            county_frames.extend(frames)

            if (i + 1) % 10 == 0:
                n = sum(len(f) for f in county_frames)
                log.info(f"  ... {i+1}/{len(price_bands)} bands, {n:,} rows")

        if county_frames:
            cdf = pd.concat(county_frames, ignore_index=True)
            log.info(f"  ✓ {county_name}: {len(cdf):,} rows")
            all_frames.append(cdf)

    return pd.concat(all_frames, ignore_index=True) if all_frames else pd.DataFrame()

In [5]:
# ---------------------------------------------------------------------------
# Clean, deduplicate, and save
# ---------------------------------------------------------------------------
def clean_and_save(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        log.error("Empty DataFrame — nothing to save.")
        return df

    # Normalize column names
    df.columns = (
        df.columns.str.strip().str.upper()
        .str.replace(r"[^A-Z0-9]+", "_", regex=True).str.strip("_")
    )

    # Deduplicate (same listing can appear in adjacent bands)
    before = len(df)
    id_cols = [c for c in ["ADDRESS", "CITY", "ZIP_OR_POSTAL_CODE"] if c in df.columns]
    if id_cols:
        df = df.drop_duplicates(subset=id_cols, keep="first")
    log.info(f"Dedup: {before:,} → {len(df):,} rows")

    # Keep residential types only
    if "PROPERTY_TYPE" in df.columns:
        keep = [
            "Single Family Residential", "Townhouse",
            "Condo/Co-op", "Multi-Family (2-4 Unit)",
        ]
        df = df[df["PROPERTY_TYPE"].isin(keep)]
        log.info(f"After property-type filter: {len(df):,}")

    # Keep only sold listings — exclude Active, Pending, Pre On-Market
    if "STATUS" in df.columns:
        before = len(df)
        df = df[df["STATUS"] == "Sold"]
        log.info(f"After STATUS filter: {len(df):,} rows (removed {before - len(df):,} non-sold)")

    # Drop null prices
    price_col = next(
        (c for c in df.columns if "PRICE" in c and "LIST" not in c and "HOA" not in c), None
    )
    if price_col:
        df = df[df[price_col].notna()]
        log.info(f"After dropping null prices: {len(df):,}")

    df.to_csv(OUTPUT_FILE, index=False)
    log.info(f"✅  Saved {len(df):,} rows → {OUTPUT_FILE}")

    # Summary
    if "COUNTY" in df.columns:
        print("\nRecords by county:")
        print(df["COUNTY"].value_counts().to_string())
    if price_col:
        p = df[price_col]
        print(f"\nPrice:  min ${p.min():,.0f}  |  median ${p.median():,.0f}  |  max ${p.max():,.0f}")

    return df

In [6]:
# ---------------------------------------------------------------------------
# Run the Redfin scraper  (≈ 15-25 min depending on rate limits)
# ---------------------------------------------------------------------------
# Uncomment the lines below to execute a fresh scrape.
# If you already have data/redfin_sold_homes.csv, skip to the Census cell.

bands = build_price_bands()
raw   = scrape_all_counties(bands)
df    = clean_and_save(raw)

2026-04-07 23:11:28,393 INFO Band plan: 71 bands × 4 counties = 284 calls (~9 min at 2.0s delay)
2026-04-07 23:11:28,393 INFO 
2026-04-07 23:11:28,394 INFO   Delaware County  (region_id=2383)
2026-04-07 23:11:28,394 INFO =======================================================
2026-04-07 23:11:31,157 INFO   ↳ Splitting $0–$200,000 → two sub-bands at $100,000
2026-04-07 23:11:36,871 INFO   ↳ Splitting $100,000–$200,000 → two sub-bands at $150,000
2026-04-07 23:11:40,266 INFO   ↳ Splitting $100,000–$150,000 → two sub-bands at $125,000
2026-04-07 23:11:49,663 INFO   ↳ Splitting $150,000–$200,000 → two sub-bands at $175,000
2026-04-07 23:11:52,695 INFO   ↳ Splitting $150,000–$175,000 → two sub-bands at $162,500
2026-04-07 23:12:02,507 INFO   ↳ Splitting $175,000–$200,000 → two sub-bands at $187,500
2026-04-07 23:12:36,691 INFO   ... 10/71 bands, 3,629 rows
2026-04-07 23:13:07,440 INFO   ... 20/71 bands, 5,298 rows
2026-04-07 23:13:38,665 INFO   ... 30/71 bands, 6,533 rows
2026-04-07 23:14:0


Records by county:
COUNTY
Montgomery County    7887
Bucks County         5886
Delaware County      5027
Chester County       4954

Price:  min $17,000  |  median $459,870  |  max $1,200,000


## Census ACS 5-Year Enrichment

Pulls socioeconomic variables at the ZCTA level and joins to the Redfin data.

**Assumption**: Census ACS 5-year estimates are stable proxies for
neighbourhood-level socioeconomics over the 1-year sold-home window.

> Set your Census API key in `.env` before running the next cell:
> ```
> CENSUS_API_KEY=your_key_here
> ```
> Get a free key at https://api.census.gov/data/key_signup.html

In [7]:
# ---------------------------------------------------------------------------
# Census ACS pull — requires `pip install census us python-dotenv`
# ---------------------------------------------------------------------------
from census import Census
import us
from dotenv import load_dotenv

load_dotenv()  # loads .env from the current working directory (or any parent)

CENSUS_API_KEY = os.environ.get("CENSUS_API_KEY", "")
if not CENSUS_API_KEY:
    raise EnvironmentError(
        "Set CENSUS_API_KEY in a .env file or as an environment variable.\n"
        "  echo 'CENSUS_API_KEY=your_key_here' > .env"
    )

REDFIN_CSV  = os.path.join(OUTPUT_DIR, "redfin_sold_homes.csv")
CENSUS_CSV  = os.path.join(OUTPUT_DIR, "census_zcta.csv")
JOINED_CSV  = os.path.join(OUTPUT_DIR, "redfin_with_census.csv")

c = Census(CENSUS_API_KEY)

ACS_VARS = {
    "B19013_001E": "median_household_income",
    "B25077_001E": "median_home_value",
    "B25064_001E": "median_gross_rent",
    "B01003_001E": "population",
    "B25002_002E": "occupied_units",
    "B25002_001E": "total_units",
    "B25035_001E": "median_year_built_neighborhood",
    "B15003_022E": "bachelors_degree_count",
    "B15003_001E": "education_total",
    "B08303_001E": "mean_commute_time",
    "B17001_002E": "poverty_count",
    "B25003_002E": "owner_occupied_units",
    "B25003_001E": "tenure_total",
}

log.info("Fetching ACS 5-year data for all US ZCTAs...")
raw = c.acs5.get(
    ["NAME"] + list(ACS_VARS.keys()),
    {"for": "zip code tabulation area:*"},
)

census_df = pd.DataFrame(raw)
census_df = census_df.rename(columns={**ACS_VARS, "zip code tabulation area": "zip"})
census_df["zip"] = census_df["zip"].astype(str).str.zfill(5)

# Filter to zips in the Redfin data
redfin_df   = pd.read_csv(REDFIN_CSV)
redfin_zips = redfin_df["ZIP_OR_POSTAL_CODE"].astype(str).str.zfill(5).str[:5].unique()
census_df   = census_df[census_df["zip"].isin(redfin_zips)]
log.info(f"Filtered to {len(census_df)} ZCTAs matching Redfin data")

# Replace Census sentinel → NaN
census_df = census_df.replace(-666666666, pd.NA)

# Derived rates
census_df["vacancy_rate"]       = 1 - census_df["occupied_units"] / census_df["total_units"]
census_df["pct_bachelors_plus"] = census_df["bachelors_degree_count"] / census_df["education_total"]
census_df["poverty_rate"]       = census_df["poverty_count"] / census_df["population"]
census_df["homeownership_rate"] = census_df["owner_occupied_units"] / census_df["tenure_total"]

# Drop raw counts (keep only rates + medians)
drop = [
    "occupied_units", "total_units", "bachelors_degree_count", "education_total",
    "poverty_count", "owner_occupied_units", "tenure_total", "NAME", "state",
]
census_df = census_df.drop(columns=[c for c in drop if c in census_df.columns])

census_df.to_csv(CENSUS_CSV, index=False)
log.info(f"Saved {CENSUS_CSV}: {len(census_df)} ZCTAs, {census_df.shape[1]} columns")
print(census_df.head())

2026-04-07 23:26:57,649 INFO Fetching ACS 5-year data for all US ZCTAs...
2026-04-07 23:27:07,488 INFO Filtered to 188 ZCTAs matching Redfin data
2026-04-07 23:27:07,493 INFO Saved data/census_zcta.csv: 188 ZCTAs, 11 columns


     median_household_income median_home_value median_gross_rent  population  \
5509                 92292.0          297700.0            1319.0      6479.0   
5697                124211.0          381700.0            1121.0     14625.0   
5701                 95052.0          286300.0            1284.0      5665.0   
5709                109613.0          408800.0            1716.0      4714.0   
5710                 90087.0          293800.0            1273.0     11947.0   

     median_year_built_neighborhood  mean_commute_time    zip  vacancy_rate  \
5509                           1977             3186.0  17527      0.018039   
5697                           1972             6219.0  18036      0.033940   
5701                           1973             2571.0  18041      0.020982   
5709                           1977             2073.0  18054      0.061614   
5710                           1966             4730.0  18055      0.044278   

      pct_bachelors_plus  poverty_rate  home

In [8]:
# ---------------------------------------------------------------------------
# Join Census → Redfin and save
# ---------------------------------------------------------------------------
redfin_df  = pd.read_csv(REDFIN_CSV)
census_df  = pd.read_csv(CENSUS_CSV)
census_df["zip"] = census_df["zip"].astype(str).str.zfill(5)
redfin_df["zip"] = redfin_df["ZIP_OR_POSTAL_CODE"].astype(str).str.zfill(5).str[:5]

# Remove duplicate columns if present
redfin_df = redfin_df.loc[:, ~redfin_df.columns.duplicated()]

joined = redfin_df.merge(census_df, on="zip", how="left")
match_rate = joined["median_household_income"].notna().mean()

log.info(f"Rows: {len(redfin_df):,} → {len(joined):,}  |  Census match: {match_rate:.1%}")

unmatched = joined[joined["median_household_income"].isna()]["zip"].unique()
if len(unmatched):
    log.warning(f"Unmatched zips ({len(unmatched)}): {sorted(unmatched)}")

joined.to_csv(JOINED_CSV, index=False)
log.info(f"Saved {JOINED_CSV}: {len(joined):,} rows, {joined.shape[1]} cols")
print(f"\nDone.  {JOINED_CSV} ready for modeling.")

2026-04-07 23:27:07,600 INFO Rows: 23,754 → 23,754  |  Census match: 99.7%
2026-04-07 23:27:07,600 WARNING Unmatched zips (21): ['18039', '18911', '18924', '18934', '18950', '18957', '18958', '18971', '18981', '18991', '19017', '19319', '19369', '19374', '19421', '19442', '19450', '19456', '19474', '19477', '19481']
2026-04-07 23:27:07,766 INFO Saved data/redfin_with_census.csv: 23,754 rows, 39 cols



Done.  data/redfin_with_census.csv ready for modeling.
